# CellWHISPER on human skin (Xenium Prime 5K)

CellWHISPER tests whether two cell types communicate through a given gene pair at points of physical contact.
For a quadruplet (cell type A, cell type B, gene 1, gene 2) it counts the neighboring A-B cell pairs in which A expresses gene 1 and B expresses gene 2 (the "whisper network"), and compares that count with its expectation under a null that shuffles cells within each cell type. The null keeps the spatial layout of every cell type, so a high z-score means co-expression at contacts beyond what the arrangement of cell types already predicts.

This notebook runs the test on a strip of epidermis and underlying dermis from the 10x Genomics
[FFPE human skin (primary dermal melanoma), Xenium Prime 5K](https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-skin) dataset.

**Data.** From the dataset page download the Xenium output bundle and keep three things in one folder:
`cell_feature_matrix.h5`, `cells.csv.gz` and `analysis.tar` (extract it; we use `analysis/clustering/gene_expression_graphclust/clusters.csv`).
Set `DATA` below to that folder. The rest of the notebook needs no other input.

**Memory.** The neighbor graph is kept as a dense boolean matrix, so the strip used here (a few tens of thousands of cells) needs a few GB of RAM.

In [ ]:
import os, sys
import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.neighbors import NearestNeighbors

sys.path.insert(0, os.path.abspath('..'))          # repo root
import cellwhisper as cw
from utils import unstack_cellwhisper, edge_segs, plot_whisper_network
import scipy.sparse as sp

# DATA = '/path/to/xenium_skin/'                     # folder with the three files above
DATA = '/scratch/users/anurendk/CellWHISPER_Data/Data/skin/'

## 1. Load and normalize

Standard scanpy preprocessing. Cluster labels from the 10x `graphclust` output are attached to `obs` so the annotation below is reproducible from the downloaded files alone.

In [ ]:
def load_xenium(folder):
    adata = sc.read_10x_h5(folder + 'cell_feature_matrix.h5')
    f_cells = folder + ('cells.csv.gz' if os.path.exists(folder + 'cells.csv.gz') else 'cells.csv')
    cells = pd.read_csv(f_cells).set_index('cell_id')
    cells.index = cells.index.astype(str)
    adata.obs = cells.loc[adata.obs_names]
    adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].to_numpy()
    cl = pd.read_csv(folder + 'analysis/clustering/gene_expression_graphclust/clusters.csv', index_col=0)
    cl.index = cl.index.astype(str)
    adata.obs['graphclust'] = cl['Cluster'].reindex(adata.obs_names).map(lambda x: str(int(x)) if pd.notna(x) else 'NA').values
    return adata

skin = load_xenium(DATA)
sc.pp.filter_cells(skin, min_genes=5); sc.pp.filter_genes(skin, min_cells=5)
skin.layers['counts'] = skin.X.copy()
sc.pp.normalize_total(skin); sc.pp.log1p(skin)
print(skin)

## 2. Region and cell-type annotation

We keep the strip with `x_centroid < 1500`, which holds the epidermis and the dermis beneath it, and skip the tumor.

Cell types come from the 10x clusters, mapped to coarse classes by their marker means. Keratinocytes are split into basal and suprabasal by a basal-marker versus suprabasal-marker score, and the suprabasal band is cut into three layers by distance from the nearest basal cell. CellWHISPER's null conditions on the cell-type labels, so in a tissue with a gradient the positional states have to be in the annotation; otherwise a quadruplet enriched in one layer is tested against the whole epidermis.

In [ ]:
ep = skin[skin.obs['x_centroid'].values < 1500].copy()
pos = ep.obsm['spatial']

def score(a, genes):
    genes = [g for g in genes if g in a.var_names]
    return a[:, genes].X.toarray().mean(axis=1)

basal = score(ep, ['TP63', 'COL17A1', 'ITGB4', 'ITGA6', 'LAMB3'])
supra = score(ep, ['DMKN', 'CDSN', 'DSG1'])

# 10x graphclust id -> coarse class (checked against marker means on this dataset)
cmap = {'20': 'KC', '26': 'KC', '27': 'KC', '28': 'KC', '5': 'KC',
        '17': 'Fibroblast', '8': 'Fibroblast',
        '19': 'Endothelial', '12': 'Endothelial', '14': 'Pericyte',
        '3': 'Immune', '15': 'Immune', '13': 'Immune', '7': 'Immune', '4': 'Immune', '21': 'Immune'}
ct = ep.obs['graphclust'].map(cmap).fillna('Other').values.astype(object)
kc = ct == 'KC'
ct[kc] = np.where(basal[kc] > supra[kc], 'Basal_KC', 'Suprabasal_KC')

# three suprabasal layers by distance to the basal layer
isb, iss = ct == 'Basal_KC', ct == 'Suprabasal_KC'
depth = NearestNeighbors(n_neighbors=1).fit(pos[isb]).kneighbors(pos[iss])[0][:, 0]
cut = np.quantile(depth, [1/3, 2/3])
layer = np.where(depth < cut[0], 'Supra_low', np.where(depth < cut[1], 'Supra_mid', 'Supra_high'))
layer[depth > 250] = 'Other'                       # keratinocyte-like cells deep in the dermis (adnexa)
ct[iss] = layer
ep.obs['cell_type'] = pd.Categorical(ct)

print(ep.shape, '| depth cuts (um):', cut.round(1))
print(ep.obs['cell_type'].value_counts().to_dict())

In [ ]:
posR = np.column_stack([pos[:, 1], -pos[:, 0]])    # rotate so the surface is at the top
annot = ep.obs['cell_type'].values.astype(str)
colors = {'Basal_KC': '#000000', 'Supra_low': '#fdae6b', 'Supra_mid': '#e6550d', 'Supra_high': '#7b3294',
          'Endothelial': '#1a9641', 'Pericyte': '#a6d96a'}

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(posR[:, 0], posR[:, 1], s=2, c='#e6e6e6', rasterized=True)
for name, col in colors.items():
    m = annot == name
    ax.scatter(posR[m, 0], posR[m, 1], s=4, c=col, rasterized=True)
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=c, label=n) for n, c in colors.items()],
          loc='upper right', frameon=False)
ax.set_aspect('equal'); ax.set_axis_off(); ax.set_title('Epidermal layers and dermal vessels')
plt.show()

## 3. One gene pair: Cx43 (GJA1) gap junctions

`run_cellwhisper_pair_adata` scores every ordered cell-type pair for one gene pair and returns matrices of whisper-network size, null mean, null standard deviation and z-score. `unstack_cellwhisper` turns that into a table. For a homotypic pair the table is symmetric and only the upper triangle is kept.

Defaults: expression is binarized at the per-gene 75th percentile, contacts are the symmetrized 5-nearest-neighbor graph, and the variance is computed in the `non_iid` mode (accounts for contacts that share a cell; `iid` treats contacts as independent and is a little faster, with the same ranking).

In [ ]:
res = cw.run_cellwhisper_pair_adata(ep, gene1='GJA1', gene2='GJA1', annot='cell_type', spatial_key='spatial',
                                    percentile=75, num_neighbor=5, mode='iid', return_graph=True)
gja1 = unstack_cellwhisper(res)
cols = ['n_whisper_edges', 'n_prox_edges', 'null_mean', 'z_score']
gja1[(gja1.z_score > 3) & (gja1.n_whisper_edges > 30)].sort_values('z_score', ascending=False)[cols].round(1)

Homotypic Cx43 contacts are significant in the outer suprabasal layer but not between basal keratinocytes, although *GJA1* is expressed throughout the basal layer. That is the point of the null: uniform co-expression within a layer is already expected from the layer's spatial arrangement and does not count as evidence.

In [ ]:
gja1.loc[['Supra_high | Supra_high | GJA1-GJA1', 'Supra_low | Supra_low | GJA1-GJA1', 'Basal_KC | Basal_KC | GJA1-GJA1'], cols].round(1)

## 4. A directional ligand-receptor pair: JAG1 (ligand) to NOTCH4 (receptor)

For a ligand-receptor pair the quadruplet is directional: gene 1 is assigned to the first cell type and gene 2 to the second, so `(Pericyte, Endothelial)` and `(Endothelial, Pericyte)` are different tests. Pass `symmetric=False` to keep all ordered pairs.

In [ ]:
res_jn = cw.run_cellwhisper_pair_adata(ep, gene1='JAG1', gene2='NOTCH4', annot='cell_type', spatial_key='spatial',
                                       percentile=75, num_neighbor=5, mode='non_iid', return_graph=True)
jn = unstack_cellwhisper(res_jn, symmetric=False)
jn[(jn.z_score > 3) & (jn.n_whisper_edges > 30)].sort_values('z_score', ascending=False)[cols].round(1)

## 5. Where the contacts are

`return_graph=True` keeps the whisper network (a sparse adjacency over all cells). `edge_segs` pulls out the edges of one quadruplet and `plot_whisper_network` draws them with a density overlay. The Cx43 circuit runs as a band along the outer epidermis; the JAG1-NOTCH4 circuit sits on dermal vessels at various depths.

In [ ]:
rot = lambda s: np.stack([s[:, :, 1], -s[:, :, 0]], axis=2)     # same rotation as posR
panels = [('Supra_high', 'Supra_high', res, gja1, 'GJA1-GJA1'),
          ('Pericyte', 'Endothelial', res_jn, jn, 'JAG1-NOTCH4')]

fig, axes = plt.subplots(2, 1, figsize=(10, 9))
for ax, (c1, c2, r, tab, g) in zip(axes, panels):
    segs = edge_segs(r['graph_whisper'], annot, c1, c2, pos)
    q = f'{c1} | {c2} | {g}'
    plot_whisper_network(ax, rot(segs), posR, title=f'{q}:  {len(segs)} whisper edges, z = {tab.loc[q, "z_score"]:.1f}')
    mb = annot == 'Basal_KC'
    ax.scatter(posR[mb, 0], posR[mb, 1], s=1.5, c='#555555', rasterized=True)   # basal layer for orientation
plt.tight_layout(); plt.show()

## 6. Scanning a database

The same call in a loop over gene pairs gives a full scan. Pairs come from any ligand-receptor resource; for the paper we used the cell-cell contact subset of CellChatDB. A loader for CellChatDB, CellPhoneDB and OmniPath, and the latent variable model that summarizes a scan into cell-type and gene preference maps, are next on the list for this repository.

```python
pairs = [('GJA1', 'GJA1'), ('JAG1', 'NOTCH4'), ('DLL4', 'NOTCH3')]
tables = [unstack_cellwhisper(cw.run_cellwhisper_pair_adata(ep, g1, g2, annot='cell_type'), symmetric=(g1 == g2))
          for g1, g2 in pairs]
scan = pd.concat(tables)
```

Parameters worth knowing: `percentile` (binarization threshold; for sparse genes the 75th percentile is 0 and "expressed" means detected), `num_neighbor` (k of the contact graph), `max_distance` (optional cap on contact distance), `mode` (`iid` or `non_iid`), and `graph_cache` (path to save the neighbor graph when running many pairs on the same cells; `_k{num_neighbor}` is appended to the name).